# QC model (BERT 2) - run 2: fix the imbalance + abstract-level eval

Meeting 2026-07-02: gold recall ~0.51 is a red flag (model leans to 'unsupported').
This run adds Fred's three asks:
1. **Diagnosis** - is it just always predicting 'no'? (gold recall split by relation type)
2. **Fix the imbalance** - reduce fp_external in training, duplicate positives, optional class weight.
3. **Abstract-level eval** - of 100%-correct abstracts how many get rejected; of faulty ones how many pass.

**How to run:** GPU runtime; upload `qc_data.zip` to `/content/`; run `SMOKE_TEST=True`
first (~3 min), then set it `False` and Run all. Results save to Drive.

## 1. Install (pinned)

In [4]:
!pip -q install "transformers==4.44.2" "datasets==2.21.0" "accelerate==0.34.2" "scikit-learn>=1.3"

## 2. Config

In [5]:
SMOKE_TEST    = False     # tiny/fast sanity run; set False for the real run
SAVE_TO_DRIVE = True     # real run: persist results+model to Google Drive

# --- imbalance experiment knobs (Fred 2026-07-02). Applied to TRAINING ONLY;
#     dev/test keep the real distribution (incl. all fp_external) as Fred wanted. ---
FP_EXTERNAL_TRAIN_FRAC = 0.25  # fraction of fp_external KEPT in training. 0.25=reduce (default), 0.0=drop, 1.0=all
DUPLICATE_POSITIVES    = 2     # replicate label==1 (supported) training rows this many times (1=off)
CLASS_WEIGHT_POS       = 1.0   # loss weight for the positive class (1.0=off; e.g. 2.0 up-weights positives)

import os, zipfile, numpy as np, pandas as pd, torch
from datasets import Dataset
from sklearn.metrics import precision_recall_fscore_support, accuracy_score, confusion_matrix
from transformers import (AutoTokenizer, AutoModelForSequenceClassification,
                          TrainingArguments, Trainer, DataCollatorWithPadding)

MODEL_NAME  = 'microsoft/BiomedNLP-BiomedBERT-base-uncased-abstract-fulltext'
MAX_LEN     = 512
EPOCHS      = 1 if SMOKE_TEST else 3
TRAIN_BATCH = 16
EVAL_BATCH  = 32
LR          = 2e-5
RECALL_FLOOR = 0.10

USE_CUDA = torch.cuda.is_available()
print('CUDA:', USE_CUDA, torch.cuda.get_device_name(0) if USE_CUDA else '(NO GPU - switch runtime to GPU!)')
LOG = []
def rec(*a):
    line = ' '.join(str(x) for x in a); print(line); LOG.append(line)

CUDA: True Tesla T4


## 3. Output location (Drive for the real run)

In [6]:
if SAVE_TO_DRIVE and not SMOKE_TEST:
    from google.colab import drive
    drive.mount('/content/drive')
    OUT = '/content/drive/MyDrive/qc_model_run2'
else:
    OUT = 'qc-out'
os.makedirs(OUT, exist_ok=True)
print('outputs ->', OUT)

Mounted at /content/drive
outputs -> /content/drive/MyDrive/qc_model_run2


## 4. Load data, build prompt, and resample TRAIN (imbalance fixes)

In [7]:
if os.path.exists('/content/qc_data.zip'):
    with zipfile.ZipFile('/content/qc_data.zip') as z: z.extractall('/content')
def load(split): return pd.read_csv(f'/content/qc_{split}.csv')
train_df, dev_df, test_df = load('train'), load('dev'), load('test')
if SMOKE_TEST:
    train_df = train_df.groupby('label', group_keys=False).sample(n=1000, random_state=42)
    dev_df   = dev_df.groupby('label', group_keys=False).sample(n=300, random_state=42)
    test_df  = test_df.groupby('label', group_keys=False).sample(n=300, random_state=42)

def make_prompt(r):
    return (f"Relation: {r.entity_a_text} -> {r.relation_type} -> {r.entity_b_text}\n"
            f"Context: {r.abstract}")
for d in (train_df, dev_df, test_df):
    d['text'] = d.apply(make_prompt, axis=1); d['label'] = d['label'].astype(int)

# --- TRAIN-ONLY resampling ---
rec('TRAIN before resample:', train_df['label'].value_counts().to_dict(),
    '| fp_external:', int((train_df['perturbation']=='fp_external').sum()))
# (a) reduce/drop fp_external
if FP_EXTERNAL_TRAIN_FRAC < 1.0:
    fp    = train_df[train_df['perturbation']=='fp_external']
    keep  = fp.sample(frac=FP_EXTERNAL_TRAIN_FRAC, random_state=42)
    train_df = pd.concat([train_df[train_df['perturbation']!='fp_external'], keep], ignore_index=True)
# (b) duplicate positive (label==1) rows
if DUPLICATE_POSITIVES > 1:
    pos = train_df[train_df['label']==1]
    train_df = pd.concat([train_df] + [pos]*(DUPLICATE_POSITIVES-1), ignore_index=True)
train_df = train_df.sample(frac=1, random_state=42).reset_index(drop=True)
dev_df   = dev_df.reset_index(drop=True); test_df = test_df.reset_index(drop=True)
rec('TRAIN after resample: ', train_df['label'].value_counts().to_dict(),
    '| pos share %.1f%%' % (100*(train_df['label']==1).mean()))
print(train_df['text'].iloc[0][:280])

TRAIN before resample: {0: 46336, 1: 8088} | fp_external: 20220
TRAIN after resample:  {0: 31171, 1: 16176} | pos share 34.2%
Relation: IL-2 receptor -> Association -> IgG
Context: The majority of CXCR5(+) PD1(+) CD4(+) T follicular helper (Tfh) cells (   >90%) are CD25(-) Bcl6(hi) , while a small subpopulation (   <10%) are CD25(+) Bcl6(low) but do not express FoxP3 and are not T regulatory cells. We p


## 5. Tokenize

In [8]:
tok = AutoTokenizer.from_pretrained(MODEL_NAME)
def to_ds(df):
    ds = Dataset.from_pandas(df[['text','label']].reset_index(drop=True))
    return ds.map(lambda b: tok(b['text'], truncation=True, max_length=MAX_LEN),
                  batched=True, remove_columns=['text'])
train_ds, dev_ds, test_ds = to_ds(train_df), to_ds(dev_df), to_ds(test_df)
collator = DataCollatorWithPadding(tok)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


Map:   0%|          | 0/47347 [00:00<?, ? examples/s]

Map:   0%|          | 0/15232 [00:00<?, ? examples/s]

Map:   0%|          | 0/15199 [00:00<?, ? examples/s]

## 6. Train

Best checkpoint by **F1** this run (not precision) so the rebalancing isn't nullified by
the trivial high-precision / near-zero-recall corner. Optional class weighting in the loss.

In [9]:
def compute_metrics(pred):
    logits, labels = pred; preds = np.argmax(logits, axis=-1)
    p, r, f1, _ = precision_recall_fscore_support(labels, preds, average='binary', pos_label=1, zero_division=0)
    return {'precision': p, 'recall': r, 'f1': f1, 'accuracy': accuracy_score(labels, preds)}

class WeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop('labels')
        outputs = model(**inputs)
        w = torch.tensor([1.0, CLASS_WEIGHT_POS], device=outputs.logits.device, dtype=outputs.logits.dtype)
        loss = torch.nn.functional.cross_entropy(outputs.logits, labels, weight=w)
        return (loss, outputs) if return_outputs else loss

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, num_labels=2,
    id2label={0:'unsupported',1:'supported'}, label2id={'unsupported':0,'supported':1})
args = TrainingArguments(
    output_dir=os.path.join(OUT,'checkpoints'), num_train_epochs=EPOCHS,
    per_device_train_batch_size=TRAIN_BATCH, per_device_eval_batch_size=EVAL_BATCH,
    learning_rate=LR, warmup_ratio=0.1, weight_decay=0.01,
    eval_strategy='epoch', save_strategy='epoch', load_best_model_at_end=True,
    metric_for_best_model='f1', greater_is_better=True,
    save_safetensors=False, save_total_limit=1,
    fp16=USE_CUDA, logging_steps=50, report_to='none')
trainer = WeightedTrainer(model=model, args=args, train_dataset=train_ds, eval_dataset=dev_ds,
                          tokenizer=tok, data_collator=collator, compute_metrics=compute_metrics)
trainer.train()
rec('DEV (argmax):', {k: round(v,4) for k,v in trainer.evaluate().items() if k.startswith('eval_')})

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at microsoft/BiomedNLP-BiomedBERT-base-uncased-abstract-fulltext and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/usr/local/lib/python3.12/dist-packages/accelerate/accelerator.py:494: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  self.scaler = torch.cuda.amp.GradScaler(**kwargs)


Epoch,Training Loss,Validation Loss,Precision,Recall,F1,Accuracy
1,0.285500,0.296405,0.568092,0.765514,0.652190,0.879070
2,0.205600,0.318220,0.691630,0.695922,0.693769,0.909007
3,0.092100,0.401429,0.687104,0.720301,0.703311,0.909992


DEV (argmax): {'eval_loss': 0.4014, 'eval_precision': 0.6871, 'eval_recall': 0.7203, 'eval_f1': 0.7033, 'eval_accuracy': 0.91, 'eval_runtime': 111.2351, 'eval_samples_per_second': 136.935, 'eval_steps_per_second': 4.279}


## 7. Predict, test @0.5, precision-first threshold

In [10]:
def probs(ds): return torch.softmax(torch.tensor(trainer.predict(ds).predictions), dim=-1)[:,1].numpy()
dev_p, test_p = probs(dev_ds), probs(test_ds)
dev_df['prob'] = dev_p; test_df['prob'] = test_p   # aligned: to_ds preserved row order
y_dev, y_test = dev_df['label'].values, test_df['label'].values
def report(y, p, thr, tag):
    pred=(p>=thr).astype(int)
    pr,rc,f1,_=precision_recall_fscore_support(y,pred,average='binary',pos_label=1,zero_division=0)
    rec(f'{tag} @thr={thr:.2f}  precision={pr:.4f}  recall={rc:.4f}  f1={f1:.4f}  acc={accuracy_score(y,pred):.4f}')
    rec('  confusion [[TN,FP],[FN,TP]] =', confusion_matrix(y,pred).tolist()); return pr,rc,f1
report(y_test, test_p, 0.5, 'TEST')
best=None
for thr in np.linspace(0.05,0.99,95):
    pred=(dev_p>=thr).astype(int)
    pr,rc,f1,_=precision_recall_fscore_support(y_dev,pred,average='binary',pos_label=1,zero_division=0)
    if rc>=RECALL_FLOOR and (best is None or pr>best[1]): best=(thr,pr,rc)
tuned_thr = best[0] if best else 0.5
rec(f'chosen threshold {tuned_thr:.2f} (dev precision={best[1]:.4f}, recall={best[2]:.4f})' if best else 'no thr met floor; 0.5')
report(y_dev,dev_p,tuned_thr,'DEV '); report(y_test,test_p,tuned_thr,'TEST')

TEST @thr=0.50  precision=0.6841  recall=0.7303  f1=0.7064  acc=0.9097
  confusion [[TN,FP],[FN,TP]] = [[12174, 763], [610, 1652]]
chosen threshold 0.99 (dev precision=0.8290, recall=0.5266)
DEV  @thr=0.99  precision=0.8290  recall=0.5266  f1=0.6441  acc=0.9138
  confusion [[TN,FP],[FN,TP]] = [[12731, 245], [1068, 1188]]
TEST @thr=0.99  precision=0.8261  recall=0.5438  f1=0.6558  acc=0.9151
  confusion [[TN,FP],[FN,TP]] = [[12678, 259], [1032, 1230]]


(0.8260577568838147, 0.5437665782493368, 0.6558251133031192)

## 8. Diagnosis: is it just predicting 'unsupported'? (item 1)

In [11]:
rec('=== TRAIN label balance (after resample) ===')
rec(train_df['label'].value_counts().to_string())
rec('=== TEST prediction distribution @0.5 (how many it calls supported) ===')
rec(pd.Series((test_p>=0.5).astype(int)).value_counts().to_string())
rec('=== gold recall split by claimed relation_type (test, @0.5) ===')
gt = test_df[test_df['label']==1].copy(); gt['pred']=(gt['prob']>=0.5).astype(int)
tab = gt.groupby('relation_type')['pred'].agg(n='size', recall='mean').round(3)
rec(tab.to_string())
rec('Read: if NoRelation recall is high but the real relations (Association/Pos/Neg) are low,')
rec('the model is leaning to "no relation" and barely learning to accept true relations.')

=== TRAIN label balance (after resample) ===
label
0    31171
1    16176
=== TEST prediction distribution @0.5 (how many it calls supported) ===
0    12784
1     2415
=== gold recall split by claimed relation_type (test, @0.5) ===
                         n  recall
relation_type                     
Association            635   0.592
Negative_Correlation   171   0.591
NoRelation            1131   0.887
Positive_Correlation   325   0.529
Read: if NoRelation recall is high but the real relations (Association/Pos/Neg) are low,
the model is leaning to "no relation" and barely learning to accept true relations.


## 9. Per-perturbation error analysis

In [12]:
for thr,tag in [(0.5,'0.5'),(tuned_thr,'tuned')]:
    rec(f'=== per-perturbation accuracy @thr={tag} ===')
    t=test_df.copy(); t['pred']=(t['prob']>=thr).astype(int); t['correct']=(t['pred']==t['label']).astype(int)
    g=t.groupby('perturbation').agg(n=('label','size'),true_label=('label','mean'),acc=('correct','mean')).round(3)
    rec(g.to_string())
    fp=t[(t['label']==0)&(t['pred']==1)]
    rec('false accepts by perturbation:'); rec(fp['perturbation'].value_counts().to_string())

=== per-perturbation accuracy @thr=0.5 ===
                   n  true_label    acc
perturbation                           
direction_swap   496         0.0  0.770
fp_external     5655         0.0  0.993
gold            2262         1.0  0.730
label_flip      6786         0.0  0.910
false accepts by perturbation:
perturbation
label_flip        612
direction_swap    114
fp_external        37
=== per-perturbation accuracy @thr=tuned ===
                   n  true_label    acc
perturbation                           
direction_swap   496         0.0  1.000
fp_external     5655         0.0  0.998
gold            2262         1.0  0.544
label_flip      6786         0.0  0.963
false accepts by perturbation:
perturbation
label_flip     249
fp_external     10


## 10. Abstract-level evaluation (item 3, Fred's main ask)

A 'correct abstract' = all its real-relation (label 1, non-NoRelation) claims; the model
should accept all. Detected errors = claims it rejects. Reject-if->=N: sweep N. A 'faulty'
abstract = the correct claims + 1 injected wrong (label 0) claim from the same abstract.

In [13]:
import random as _r
REAL = ['Association','Positive_Correlation','Negative_Correlation']
def abstract_level(thr, tag):
    rec(f'=== abstract-level @thr={tag} ===')
    t = test_df.copy(); t['pred'] = (t['prob'] >= thr).astype(int)
    gold = t[(t['label']==1) & (t['relation_type'].isin(REAL))]
    per = gold.groupby('pmid')['pred'].agg(n='size', detected=lambda s:int((s==0).sum()))
    if len(per)==0: rec('  (no real-relation gold abstracts)'); return
    rec(f'  correct abstracts: {len(per)} | median claims/abstract {int(per["n"].median())}')
    for N in [1,2,3]:
        rec(f'  reject if >= {N} detected error(s): correct abstracts PASS = {(per["detected"]<N).mean():.3f} (higher better)')
    rng=_r.Random(42)
    neg_by_pmid = {p:g.index.tolist() for p,g in t[t['label']==0].groupby('pmid')}
    det=[]
    for pmid,g in gold.groupby('pmid'):
        if pmid not in neg_by_pmid: continue
        inj = rng.choice(neg_by_pmid[pmid])
        claims = g['pred'].tolist() + [int(t.loc[inj,'pred'])]
        det.append(sum(1 for c in claims if c==0))
    if det:
        det=np.array(det)
        rec(f'  faulty abstracts (correct + 1 injected error): {len(det)}')
        for N in [1,2,3]:
            rec(f'  reject if >= {N} detected error(s): faulty abstracts PASS = {(det<N).mean():.3f} (lower better)')
abstract_level(0.5,'0.5'); abstract_level(tuned_thr,'tuned')

=== abstract-level @thr=0.5 ===
  correct abstracts: 100 | median claims/abstract 7
  reject if >= 1 detected error(s): correct abstracts PASS = 0.190 (higher better)
  reject if >= 2 detected error(s): correct abstracts PASS = 0.390 (higher better)
  reject if >= 3 detected error(s): correct abstracts PASS = 0.490 (higher better)
  faulty abstracts (correct + 1 injected error): 100
  reject if >= 1 detected error(s): faulty abstracts PASS = 0.000 (lower better)
  reject if >= 2 detected error(s): faulty abstracts PASS = 0.190 (lower better)
  reject if >= 3 detected error(s): faulty abstracts PASS = 0.400 (lower better)
=== abstract-level @thr=tuned ===
  correct abstracts: 100 | median claims/abstract 7
  reject if >= 1 detected error(s): correct abstracts PASS = 0.070 (higher better)
  reject if >= 2 detected error(s): correct abstracts PASS = 0.100 (higher better)
  reject if >= 3 detected error(s): correct abstracts PASS = 0.240 (higher better)
  faulty abstracts (correct + 1 inje

## 11. Save

In [14]:
with open(os.path.join(OUT,'results.txt'),'w') as f: f.write('\n'.join(LOG))
print('wrote', os.path.join(OUT,'results.txt'))
if not SMOKE_TEST:
    mp=os.path.join(OUT,'qc-pubmedbert-final'); trainer.save_model(mp); tok.save_pretrained(mp)
    print('model saved ->', mp, '| everything in MyDrive/qc_model_run2/')
else:
    print('Smoke test OK. Set SMOKE_TEST=False and Run all.')

wrote /content/drive/MyDrive/qc_model_run2/results.txt
model saved -> /content/drive/MyDrive/qc_model_run2/qc-pubmedbert-final | everything in MyDrive/qc_model_run2/
